# Mantenimiento Predictivo con NASA CMAPSS

Este notebook implementa un pipeline completo de **mantenimiento predictivo** usando el dataset CMAPSS (Commercial Modular Aero-Propulsion System Simulation) de la NASA.

## Dataset
- **Archivo**: `train_FD001.txt` y `RUL_FD001.txt`
- **Descripción**: Datos de sensores de motores de avión hasta el fallo.
- **Descarga**: [NASA Prognostics Center](https://www.nasa.gov/intelligent-systems-division/discovery-and-systems-health/pcoe/pcoe-data-set-repository/)

## Estructura del notebook
1. Imports y configuración
2. Funciones auxiliares
3. Carga y preprocesamiento de datos
4. Análisis exploratorio con PCA
5. Modelado de series de tiempo (ARIMA)
6. Detección de anomalías (Isolation Forest)
7. Clasificación con Random Forest
8. Clasificación con LSTM
9. LSTM + Atención
10. Comparación por motor

## 1. Imports y configuración

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score, f1_score

import statsmodels.api as sm

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import LSTM, Dense, Input, Layer
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Reproducibilidad
np.random.seed(42)
tf.random.set_seed(42)

%matplotlib inline
plt.rcParams['figure.dpi'] = 100

## 2. Funciones auxiliares

### 2.1 Helpers de visualización

In [ ]:
def plot_hist(data, bins, title, xlabel, ylabel):
    plt.figure(figsize=(8, 4))
    plt.hist(data, bins=bins, alpha=0.7)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.show()


def plot_line(y, title, xlabel, ylabel, x=None):
    plt.figure(figsize=(8, 4))
    if x is None:
        plt.plot(y, alpha=0.8)
    else:
        plt.plot(x, y, alpha=0.8)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.show()


def plot_two_lines(x, y1, y2, labels, title, xlabel, ylabel, styles=None, hline=None):
    plt.figure(figsize=(10, 4))
    style1 = '-'  if not styles else styles[0]
    style2 = '--' if not styles else styles[1]
    plt.plot(x, y1, label=labels[0], linestyle=style1)
    plt.plot(x, y2, label=labels[1], linestyle=style2)
    if hline is not None:
        plt.axhline(hline, color='red', linestyle=':', linewidth=0.8, label='Threshold')
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.legend()
    plt.tight_layout()
    plt.show()

### 2.2 Construcción de secuencias y splits

In [ ]:
def make_sequences(df, sensor, window=30):
    """Construye ventanas deslizantes univariadas para un sensor dado."""
    sequences, labels, units = [], [], []
    for unit_id, group in df.groupby('unit'):
        values  = group[sensor].values
        targets = group['distress'].values
        for i in range(len(values) - window):
            sequences.append(values[i:i + window])
            labels.append(targets[i + window])
            units.append(unit_id)
    return np.array(sequences), np.array(labels), np.array(units)


def make_lstm_sequences(df, sensor_cols, window=30):
    """Construye ventanas deslizantes multivariadas para LSTM."""
    sequences, labels, units = [], [], []
    for unit_id, group in df.groupby('unit'):
        values  = group[sensor_cols].values
        targets = group['distress'].values
        for i in range(len(values) - window):
            sequences.append(values[i:i + window])
            labels.append(targets[i + window])
            units.append(unit_id)
    return np.array(sequences), np.array(labels), np.array(units)


def unit_time_series_splits(units: np.ndarray, n_splits: int = 5):
    """
    Divide cronológicamente por unidad (motor).
    El fold k usa los motores más tempranos para train y el siguiente bloque para test.
    """
    unique_units = np.array(sorted(pd.unique(units)))
    tscv = TimeSeriesSplit(n_splits=n_splits)
    for train_u_idx, test_u_idx in tscv.split(unique_units):
        train_units = set(unique_units[train_u_idx])
        test_units  = set(unique_units[test_u_idx])
        train_idx = np.where(np.isin(units, list(train_units)))[0]
        test_idx  = np.where(np.isin(units, list(test_units)))[0]
        yield train_idx, test_idx


def compute_metrics(y_true, y_prob):
    """Calcula accuracy, ROC-AUC, PR-AUC y F1 para un clasificador binario."""
    y_hat = (y_prob >= 0.5).astype(int)
    return {
        'accuracy': float(accuracy_score(y_true, y_hat)),
        'roc_auc':  float(roc_auc_score(y_true, y_prob))
                    if len(np.unique(y_true)) > 1 else np.nan,
        'pr_auc':   float(average_precision_score(y_true, y_prob))
                    if len(np.unique(y_true)) > 1 else np.nan,
        'f1':       float(f1_score(y_true, y_hat))
                    if len(np.unique(y_hat)) > 1 else np.nan,
    }

## 3. Carga y preprocesamiento de datos

El dataset FD001 contiene:
- **unit**: ID del motor
- **time**: ciclo operacional
- **op_setting_1/2/3**: condiciones de operación
- **sensor_1 … sensor_21**: lecturas de 21 sensores

La etiqueta objetivo es **distress**: `True` si el motor tiene menos de 20 ciclos de vida restante (RUL < 20).

In [ ]:
df = pd.read_csv("train_FD001.txt", sep=r"\s+", header=None)
df.dropna(axis=1, inplace=True)
df.columns = (
    ['unit', 'time', 'op_setting_1', 'op_setting_2', 'op_setting_3']
    + [f'sensor_{i}' for i in range(1, 22)]
)

# RUL ground-truth (usado como referencia; no se incluye en features)
rul = pd.read_csv("RUL_FD001.txt", header=None)
rul.columns = ['max_RUL']
rul['unit'] = rul.index + 1

# Calcular RUL dentro del set de entrenamiento
last_cycle = df.groupby('unit')['time'].max().reset_index()
last_cycle.columns = ['unit', 'max_time']
df = df.merge(last_cycle, on='unit')
df['RUL']     = df['max_time'] - df['time']
df['distress'] = df['RUL'] < 20

print(f"Filas: {len(df):,}  |  Motores: {df['unit'].nunique()}")
print(f"Proporción en distress: {df['distress'].mean():.2%}")
df.head()

In [ ]:
# Sensores con mayor variabilidad informativa (selección manual basada en literatura)
selected_sensors = ['sensor_9', 'sensor_14', 'sensor_4', 'sensor_3', 'sensor_17', 'sensor_2']

# Distribución de RUL
plot_hist(
    df['RUL'], bins=50,
    title='Distribución de Vida Útil Restante (RUL)',
    xlabel='RUL (ciclos)', ylabel='Frecuencia'
)

## 4. Análisis con PCA

Reducimos la dimensionalidad de los sensores a 3 componentes principales.
El scaler y la PCA se ajustan **solo sobre datos saludables** (primeros 30 ciclos)
para evitar que el deterioro influya en la definición de la zona normal.

In [ ]:
healthy_mask = df['time'] <= 30

scaler_vis = StandardScaler().fit(df.loc[healthy_mask, selected_sensors])
X_scaled_vis = scaler_vis.transform(df[selected_sensors])

pca_vis = PCA(n_components=3).fit(X_scaled_vis[healthy_mask.values])
pca_factors_vis = pca_vis.transform(X_scaled_vis)
df[['pca_1', 'pca_2', 'pca_3']] = pca_factors_vis

print(f"Varianza explicada por PC1-PC3: {pca_vis.explained_variance_ratio_.cumsum()[-1]:.2%}")

In [ ]:
# Nube de operación saludable en espacio PCA
healthy_df = df[df['time'] <= 30]
X_pca_healthy = healthy_df[['pca_1', 'pca_2']].values

plt.figure(figsize=(7, 5))
plt.scatter(X_pca_healthy[:, 0], X_pca_healthy[:, 1], alpha=0.3, s=10)
plt.title('PCA — Zona de operación saludable (primeros 30 ciclos)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.tight_layout()
plt.show()

In [ ]:
# Distancia de Mahalanobis simplificada (distancia euclidiana al centroide saludable)
center = X_pca_healthy.mean(axis=0)
df['pca_distance'] = np.linalg.norm(df[['pca_1', 'pca_2']].values - center, axis=1)

threshold = df['pca_distance'].mean() + 3 * df['pca_distance'].std()
df['pca_anomaly'] = df['pca_distance'] > threshold

print(f"Umbral PCA: {threshold:.3f}")
print(f"Anomalías detectadas: {df['pca_anomaly'].sum():,} ({df['pca_anomaly'].mean():.2%})")

## 5. Modelado de series de tiempo con ARIMA

Ajustamos un ARIMA(1,1,1) sobre `pca_1` del motor 1 y analizamos
los residuos para detectar comportamiento anómalo.

In [ ]:
unit_df = df[df['unit'] == 1].copy()

arima_model = sm.tsa.ARIMA(unit_df['pca_1'], order=(1, 1, 1))
fit = arima_model.fit()
forecast  = fit.predict(start=1, end=len(unit_df), dynamic=False)
residuals = unit_df['pca_1'].iloc[1:].values - forecast[1:]

print(fit.summary())

In [ ]:
sigma = residuals.std()

plt.figure(figsize=(10, 4))
plt.plot(residuals, alpha=0.7, label='Residuos')
plt.axhline( 2 * sigma, color='r', linestyle='--', label='+2σ')
plt.axhline(-2 * sigma, color='r', linestyle='--', label='-2σ')
plt.title('Residuos ARIMA sobre PC1 — Motor 1')
plt.xlabel('Ciclo')
plt.ylabel('Residuo')
plt.legend()
plt.tight_layout()
plt.show()

## 6. Detección de anomalías — Isolation Forest

Isolation Forest es un algoritmo no supervisado que aísla observaciones
anómalas usando árboles de decisión aleatorios. Lo aplicamos sobre las
3 componentes PCA.

In [ ]:
pca_features = df[['pca_1', 'pca_2', 'pca_3']].values

clf = IsolationForest(contamination=0.05, random_state=42)
df['anomaly_iforest'] = clf.fit_predict(pca_features) == -1

print(f"Anomalías Isolation Forest: {df['anomaly_iforest'].sum():,} ({df['anomaly_iforest'].mean():.2%})")

In [ ]:
plt.figure(figsize=(9, 4))
plt.hist(df['pca_distance'], bins=50, alpha=0.6, label='Distancia PCA')
plt.axvline(threshold, color='red', linestyle='--', label=f'Umbral ({threshold:.2f})')
plt.title('Distribución de distancia PCA al centroide saludable')
plt.xlabel('Distancia')
plt.ylabel('Frecuencia')
plt.legend()
plt.tight_layout()
plt.show()

## 7. Clasificación de secuencias — Random Forest

Construimos ventanas de 30 ciclos sobre `sensor_9` y usamos
**TimeSeriesSplit por unidad** para evitar data leakage entre motores.

In [ ]:
X_seq, y_seq, units_rf = make_sequences(df, 'sensor_9', window=30)
print(f"Secuencias: {X_seq.shape}  |  Positivos (distress): {y_seq.sum():,} ({y_seq.mean():.2%})")

In [ ]:
rf_metrics = []

for fold, (train_idx, test_idx) in enumerate(
    unit_time_series_splits(units_rf, n_splits=5), start=1
):
    X_train_rf, X_test_rf = X_seq[train_idx], X_seq[test_idx]
    y_train_rf, y_test_rf = y_seq[train_idx], y_seq[test_idx]

    rf_clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_clf.fit(X_train_rf, y_train_rf)

    y_pred_rf = rf_clf.predict_proba(X_test_rf)[:, 1]
    metrics   = compute_metrics(y_test_rf, y_pred_rf)
    rf_metrics.append(metrics)
    print(f"Fold {fold}: {metrics}")

rf_mean = {k: np.nanmean([m[k] for m in rf_metrics]) for k in rf_metrics[0]}
print(f"\nMedia RF: {rf_mean}")

In [ ]:
plot_hist(
    y_pred_rf, bins=30,
    title='Random Forest — Probabilidades de distress (último fold)',
    xlabel='Probabilidad predicha', ylabel='Frecuencia'
)

## 8. Clasificación con LSTM

Extendemos el enfoque anterior a una red LSTM multivariada:
- Entrada: ventana de 30 ciclos × 6 sensores → reducida a 3 PCs por fold
- Salida: probabilidad de distress

> **Importante**: scaler y PCA se ajustan **solo sobre datos de train** de cada fold.

In [ ]:
X, y, units_lstm = make_lstm_sequences(df, selected_sensors, window=30)
n_features = len(selected_sensors)
print(f"Secuencias LSTM: {X.shape}")

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=0),
]

lstm_metrics = []

for fold, (train_idx, test_idx) in enumerate(
    unit_time_series_splits(units_lstm, n_splits=5), start=1
):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Normalización y PCA: ajustar SOLO en train
    scaler_lstm = StandardScaler().fit(X_train.reshape(-1, n_features))
    X_train_sc  = scaler_lstm.transform(X_train.reshape(-1, n_features)).reshape(X_train.shape)
    X_test_sc   = scaler_lstm.transform(X_test.reshape(-1, n_features)).reshape(X_test.shape)

    pca_lstm   = PCA(n_components=3).fit(X_train_sc.reshape(-1, n_features))
    X_train_seq = pca_lstm.transform(X_train_sc.reshape(-1, n_features)).reshape(X_train.shape[0], X_train.shape[1], 3)
    X_test_seq  = pca_lstm.transform(X_test_sc.reshape(-1, n_features)).reshape(X_test.shape[0], X_test.shape[1], 3)

    # Modelo LSTM
    model_lstm = Sequential([
        LSTM(64, input_shape=(X_train_seq.shape[1], 3)),
        Dense(1, activation='sigmoid')
    ])
    model_lstm.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    model_lstm.fit(
        X_train_seq, y_train,
        epochs=20, batch_size=32, shuffle=False,
        validation_data=(X_test_seq, y_test),
        callbacks=callbacks, verbose=0
    )

    y_pred_lstm = model_lstm.predict(X_test_seq, verbose=0).flatten()
    metrics = compute_metrics(y_test, y_pred_lstm)
    lstm_metrics.append(metrics)
    print(f"Fold {fold}: {metrics}")

lstm_mean = {k: np.nanmean([m[k] for m in lstm_metrics]) for k in lstm_metrics[0]}
print(f"\nMedia LSTM: {lstm_mean}")

In [ ]:
plot_line(
    y_pred_lstm,
    title='LSTM — Probabilidades de distress (último fold)',
    xlabel='Índice de muestra', ylabel='Probabilidad predicha'
)

## 9. LSTM + Mecanismo de Atención

El mecanismo de atención aprende a **ponderar los pasos de tiempo** más
relevantes para la predicción de distress, añadiendo interpretabilidad.

In [ ]:
class AttentionWithWeights(Layer):
    """Capa de atención con pesos observables para visualización."""

    def build(self, input_shape):
        self.W = self.add_weight(
            name='attn_weight',
            shape=(input_shape[-1], 1),
            initializer='random_normal',
            trainable=True
        )
        super().build(input_shape)

    def call(self, inputs):
        scores  = tf.matmul(inputs, self.W)          # (batch, timesteps, 1)
        weights = tf.nn.softmax(scores, axis=1)      # normalización temporal
        context = tf.reduce_sum(inputs * weights, axis=1)  # suma ponderada
        return context, weights

In [ ]:
# Entrenamos el modelo de atención sobre el último fold (para visualización consistente)
input_seq  = Input(shape=(X_train_seq.shape[1], 3))
lstm_out   = LSTM(64, return_sequences=True)(input_seq)
context, attn_weights = AttentionWithWeights()(lstm_out)
output     = Dense(1, activation='sigmoid', name='pred')(context)

model_attn = Model(inputs=input_seq, outputs={'pred': output, 'attn': attn_weights})
model_attn.compile(
    optimizer='adam',
    loss={'pred': 'binary_crossentropy'},
    metrics={'pred': 'accuracy'}
)
model_attn.summary()

In [ ]:
model_attn.fit(
    X_train_seq, {'pred': y_train},
    epochs=10, batch_size=32, shuffle=False,
    validation_data=(X_test_seq, {'pred': y_test}),
    callbacks=callbacks, verbose=1
)

## 10. Comparación por motor

Visualizamos las predicciones de LSTM y LSTM+Atención para un motor específico
a lo largo de toda su vida operacional.

In [ ]:
UNIT_ID = 3
WINDOW  = 30

engine_data = df[df['unit'] == UNIT_ID].copy()

# Construir secuencias del motor
X_engine_raw = np.array([
    engine_data[selected_sensors].values[i:i + WINDOW]
    for i in range(len(engine_data) - WINDOW)
])
X_engine_sc = scaler_lstm.transform(
    X_engine_raw.reshape(-1, n_features)
).reshape(X_engine_raw.shape)

X_engine = pca_lstm.transform(
    X_engine_sc.reshape(-1, n_features)
).reshape(X_engine_raw.shape[0], WINDOW, 3)

cycles = engine_data['time'].values[WINDOW:]
print(f"Motor {UNIT_ID}: {len(cycles)} puntos de predicción")

In [ ]:
pred_lstm_engine  = model_lstm.predict(X_engine, verbose=0).flatten()
pred_dict         = model_attn.predict(X_engine, verbose=0)
pred_attn_engine  = pred_dict['pred'].flatten()

# Gráficas individuales
plot_line(
    pred_lstm_engine,
    title=f'LSTM — Probabilidad de distress, Motor {UNIT_ID}',
    xlabel='Ciclo', ylabel='Probabilidad',
    x=cycles
)
plot_line(
    pred_attn_engine,
    title=f'LSTM + Atención — Probabilidad de distress, Motor {UNIT_ID}',
    xlabel='Ciclo', ylabel='Probabilidad',
    x=cycles
)

In [ ]:
plot_two_lines(
    x=cycles,
    y1=pred_lstm_engine,
    y2=pred_attn_engine,
    labels=['LSTM', 'LSTM + Atención'],
    title=f'Motor {UNIT_ID} — LSTM vs LSTM + Atención',
    xlabel='Ciclo', ylabel='Probabilidad de distress',
    styles=['-', '--'],
    hline=0.5
)

## 11. Resumen de métricas

Comparamos los dos modelos supervisados sobre los 5 folds.

In [ ]:
summary = pd.DataFrame({
    'Modelo': ['Random Forest', 'LSTM'],
    'Accuracy':  [rf_mean['accuracy'],  lstm_mean['accuracy']],
    'ROC-AUC':   [rf_mean['roc_auc'],   lstm_mean['roc_auc']],
    'PR-AUC':    [rf_mean['pr_auc'],    lstm_mean['pr_auc']],
    'F1':        [rf_mean['f1'],        lstm_mean['f1']],
})
summary.set_index('Modelo', inplace=True)
summary.round(4)